# Tracking + play-by-play exploration

Look at raw rows from a tracking Parquet and the play-by-play Parquet for the same game, side by side. Exploration only; nothing here is a pipeline dependency (see PROJECT.md).

Run this with the project's `.venv` as the kernel so `nba_assists` is already importable (`.venv/bin/pip install -e ".[dev]"`).

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd

from nba_assists.config import get_config

cfg = get_config()
tracking_dir = cfg.get_path("tracking")
pbp_dir = cfg.get_path("pbp")
pd.set_option("display.max_columns", None)

## Pick a game

Any `game_id` that has both tracking and play-by-play works. Change `GAME_ID` to look at a different one.

In [ ]:
con = duckdb.connect()
GAME_ID = "0021500052" #con.execute(
#     f"SELECT DISTINCT game_id FROM read_parquet('{tracking_dir}/*.parquet') ORDER BY game_id LIMIT 1"
# ).fetchone()[0]

tracking_path = Path(
    con.execute(
        f"SELECT filename FROM read_parquet('{tracking_dir}/*.parquet', filename=true) "
        "WHERE game_id = ? LIMIT 1",
        [GAME_ID],
    ).fetchone()[0]
)
con.close()

print("game_id:", GAME_ID)
print("tracking file:", tracking_path)

## Tracking rows

One row per tracked object per frame (`ts_ms`). A full frame has 10 player rows plus 1 ball row (`team_id`/`player_id` equal to the ball sentinels in `config.yaml`).

In [ ]:
tracking = pd.read_parquet(tracking_path)
print(tracking.shape)
tracking.head(15)

In [ ]:
# every row sharing the first frame's ts_ms -- should be 10 players + 1 ball
first_ts = tracking["ts_ms"].iloc[0]
tracking[tracking["ts_ms"] == first_ts]

## Play-by-play rows, same game

Already sorted by `PERIOD`, then `clock_s` descending (see `scripts/prepare_pbp.py`).

In [ ]:
pbp = pd.read_parquet(pbp_dir / "pbp_2015_16.parquet")
game_pbp = pbp[pbp["GAME_ID"] == GAME_ID].reset_index(drop=True)
print(game_pbp.shape)
game_pbp.head(15)

## Cross-referencing one event

Take the game's first made shot from play-by-play, then look at the tracking ball frames around its recorded `clock_s` -- the same kind of window `scripts/detect_makes.py` searches (see `rim_detection:` in `config.yaml`).

In [ ]:
made_shot_type = int(cfg.pbp.event_types.made_shot)
make_row = game_pbp[game_pbp["EVENTMSGTYPE"] == made_shot_type].iloc[0]
make_row[["EVENTNUM", "PERIOD", "clock_s", "HOMEDESCRIPTION", "VISITORDESCRIPTION"]]

In [ ]:
period = int(make_row["PERIOD"])
clock_s = float(make_row["clock_s"])
ball_team_id = int(cfg.sentinels.ball_team_id)
ball_player_id = int(cfg.sentinels.ball_player_id)

nearby_ball_frames = tracking[
    (tracking["quarter"] == period)
    & (tracking["team_id"] == ball_team_id)
    & (tracking["player_id"] == ball_player_id)
    & tracking["game_clock"].between(clock_s - 2, clock_s + 2)
].sort_values("ts_ms")[["ts_ms", "game_clock", "x", "y", "z"]]
nearby_ball_frames

## Tracking coverage across the game

SportVU moments are stitched together per play-by-play event, so there are real gaps in frame coverage even while the game clock is running (see PROJECT.md, "Tracking frames"). This ribbon shows, across the whole game (quarter 1 tipoff to the final buzzer), which ~1-second slices of game time have at least one tracking frame (blue) and which don't (gray track).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# PROJECT.md, "Clock": "A regulation period is 720 s; an overtime period is 300 s."
REGULATION_PERIOD_S = 720.0
OVERTIME_PERIOD_S = 300.0
COVERAGE_BIN_S = 1.0  # resolution of the coverage ribbon below


def period_length_s(quarter: int) -> float:
    return REGULATION_PERIOD_S if quarter <= 4 else OVERTIME_PERIOD_S


def runs_s(mask, bin_s: float):
    """Start/end times (seconds) of each contiguous True run in a bool mask."""
    padded = np.concatenate(([False], mask, [False]))
    edges = np.flatnonzero(np.diff(padded))
    return edges[0::2] * bin_s, edges[1::2] * bin_s


ball = tracking[(tracking["team_id"] == ball_team_id) & (tracking["player_id"] == ball_player_id)]

quarters = sorted(int(q) for q in tracking["quarter"].unique())
quarter_start_s = {}
offset = 0.0
for q in quarters:
    quarter_start_s[q] = offset
    offset += period_length_s(q)
total_game_length_s = offset

n_bins = int(total_game_length_s // COVERAGE_BIN_S)
covered = np.zeros(n_bins, dtype=bool)
for q in quarters:
    elapsed_in_q = period_length_s(q) - ball.loc[ball["quarter"] == q, "game_clock"].to_numpy()
    elapsed = quarter_start_s[q] + elapsed_in_q
    bin_idx = np.clip((elapsed // COVERAGE_BIN_S).astype(int), 0, n_bins - 1)
    covered[bin_idx] = True

covered_starts_s, covered_ends_s = runs_s(covered, COVERAGE_BIN_S)
covered_segments_min = list(zip(covered_starts_s / 60, (covered_ends_s - covered_starts_s) / 60))

fig, ax = plt.subplots(figsize=(12, 1.8))
ax.broken_barh([(0, total_game_length_s / 60)], (0.2, 0.6), facecolors="#e1e0d9")  # gap track
ax.broken_barh(covered_segments_min, (0.2, 0.6), facecolors="#2a78d6")  # covered

for q in quarters[1:]:
    ax.axvline(quarter_start_s[q] / 60, color="#898781", linestyle="--", linewidth=1)

ax.set_xlim(0, total_game_length_s / 60)
ax.set_ylim(0, 1)
ax.set_yticks([])
for spine in ("top", "right", "left"):
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color("#c3c2b7")
ax.tick_params(axis="x", colors="#52514e")
ax.set_xlabel("elapsed game time (minutes) -- dashed lines mark quarter boundaries", color="#52514e")
ax.set_title(f"Tracking frame coverage -- game {GAME_ID}", color="#0b0b0b")

print(f"coverage: {covered.mean():.1%} of {COVERAGE_BIN_S:g}s game-time bins have at least one frame")
plt.show()

### Largest gaps (table view of the same ribbon)

In [ ]:
gap_starts_s, gap_ends_s = runs_s(~covered, COVERAGE_BIN_S)
gaps = pd.DataFrame(
    {
        "start_min": gap_starts_s / 60,
        "end_min": gap_ends_s / 60,
        "duration_s": gap_ends_s - gap_starts_s,
    }
).sort_values("duration_s", ascending=False).reset_index(drop=True)
gaps.head(15)